# TSR Colab Production-Lite Demo

Notebook này là bản demo **Google Colab-ready** cho repo `adas-tsr`.

Mục tiêu:
- chạy được end-to-end trên Colab hoặc local Jupyter;
- dùng lại baseline detector `best.pt` khi có thể;
- bổ sung các khối **production-lite** có thể mô phỏng trên notebook: `Image Quality Gate`, `ODD gate`, `tracking + temporal confirmation`, `sign lifecycle`, `warning level`, `HMI policy`, `JSONL event log`;
- tạo video đầu ra annotated để review đầy đủ hành vi của feature.

Đầu ra chính của notebook:
- video annotated;
- file `events.jsonl` để replay;
- bảng tóm tắt runtime, quality, state, warning và publish decisions;
- cụm KPI/proxy metric để đọc ODD, confidence và size coverage ngay trong notebook.


## Scope của bản demo này

### Có thể triển khai được ngay trên Colab
- Setup runtime và kiểm tra RAM/GPU.
- Nạp model YOLO từ repo local hoặc Hugging Face.
- Chạy detector trên toàn bộ video đầu vào.
- Thêm `quality gate` dựa trên blur / brightness / glare heuristics.
- Thêm `ODD phase-1 gate` ở mức demo.
- Thêm `tracking + temporal confirmation` đơn giản theo IoU.
- Thêm `sign lifecycle` với các state `CANDIDATE`, `CONFIRMED`, `ACTIVE`, `STALE`, `SUPPRESSED`, `EXPIRED`.
- Thêm `warning level` và `HMI policy` đơn giản.
- Ghi `JSONL event log` để giải thích vì sao sign được publish hay bị suppress.

### Chưa thể coi là production thật
- Không có lane model thật.
- Không có HD map hoặc localization confidence thật.
- Không có diagnostics ECU, watchdog thật, DTC thật.
- Không có benchmark chuẩn ECU hoặc safety case.

Vì vậy notebook này nên được xem là **production-inspired demo**, không phải production implementation.


## Mermaid renderer dùng Python

Notebook này giữ Mermaid source ngay trong notebook và render **inline** tại chỗ cần hiển thị.

Cấu trúc mới:
- một helper `display_mermaid()` dùng chung;
- một cụm cell liên tiếp chỉ chứa Mermaid source cho từng sơ đồ;
- ở mỗi mục nội dung chỉ còn một cell ngắn gọi `display_mermaid(...)`.

Ưu điểm:
- Mermaid source tập trung, dễ sửa;
- không cần export `.svg` ra file ngoài;
- nội dung notebook gọn hơn, ít logic phụ hơn.


In [ ]:
# Mermaid inline renderer for Colab / Jupyter
import importlib.util
import subprocess
import sys
from html import escape

if importlib.util.find_spec('requests') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'requests>=2.31.0'])

import requests
from IPython.display import HTML, display


def mermaid_to_svg(mermaid_text: str, timeout: int = 30) -> str:
    # Render Mermaid -> SVG qua HTTP API và trả về string SVG.
    # Không ghi file ra ngoài notebook.
    resp = requests.post(
        'https://kroki.io/mermaid/svg',
        data=mermaid_text.encode('utf-8'),
        timeout=timeout,
    )
    resp.raise_for_status()
    return resp.text


def display_mermaid(mermaid_text: str, caption: str | None = None) -> None:
    svg = mermaid_to_svg(mermaid_text)
    caption_html = ''
    if caption:
        caption_html = f'<div style="font-size:12px;color:#666;margin-top:8px;">{escape(caption)}</div>'
    display(HTML(svg + caption_html))


## Mermaid source tập trung

Các cell dưới đây chỉ chứa Mermaid source. Khi cần chỉnh sơ đồ, bạn sửa ngay tại đây rồi chạy lại cell render tương ứng ở phần nội dung.


In [ ]:
# Mermaid source: Bước 1 - Setup runtime và dependency
diagram_setup_runtime = r'''flowchart LR
    A[Notebook Start] --> B[Detect Colab or Local]
    B --> C[Check required packages]
    C --> D[Install missing packages]
    D --> E[Print runtime summary]
'''


In [ ]:
# Mermaid source: Bước 2 - Chọn profile theo RAM và nguồn video đầu vào
diagram_profile_input = r'''flowchart TD
    A[Read system RAM] --> B{RAM tier}
    B -->|High| C[Profile high]
    B -->|Medium| D[Profile medium]
    B -->|Low| E[Profile low]
    C --> F[Resolve input video]
    D --> F
    E --> F
    F --> G[Create output directory]
'''


In [ ]:
# Mermaid source: Kiến trúc production-lite được mô phỏng trong notebook
diagram_production_lite_arch = r'''flowchart LR
    A[Camera Frame] --> B[Image Quality Gate]
    B --> C[ODD Gate]
    C --> D[YOLO Detector]
    D --> E[Tracking and Temporal Confirmation]
    E --> F[Sign Lifecycle]
    F --> G[Warning Level and HMI Policy]
    G --> H[Annotated Video]
    G --> I[JSONL Event Log]
'''


In [ ]:
# Mermaid source: Bước 3 - Helper layer cho perception và overlay
diagram_helper_layer = r'''flowchart LR
    A[Raw frame] --> B[preprocess]
    B --> C[quality assessment]
    C --> D[YOLO detections]
    D --> E[bbox scaling]
    E --> F[family classification]
    F --> G[warning level mapping]
    G --> H[overlay rendering]
'''


In [ ]:
# Mermaid source: Bước 4 - Tracking, temporal confirmation và state machine
diagram_tracking_state = r'''flowchart LR
    A[Detections per frame] --> B[Association by IoU and family]
    B --> C[Update hit miss counters]
    C --> D[State transition]
    D --> E[Choose primary sign]
    E --> F[Apply warning level and HMI policy]
'''


In [ ]:
# Mermaid source: Bước 5 - Nạp model detector
diagram_load_model = r'''flowchart LR
    A[Check local weights] --> B{best.pt exists?}
    B -->|Yes| C[Use local model]
    B -->|No| D[Download from Hugging Face]
    C --> E[Load YOLO]
    D --> E
'''


In [ ]:
# Mermaid source: Bước 6 - Chạy production-lite pipeline end-to-end
diagram_end_to_end = r'''flowchart TD
    A[Open video] --> B[Read frame]
    B --> C[Preprocess]
    C --> D[Quality gate]
    D --> E[ODD gate]
    E --> F{Should infer?}
    F -->|Yes| G[YOLO detect]
    F -->|No| H[Skip inference]
    G --> I[Scale detections]
    H --> I
    I --> J[Update tracks]
    J --> K[Choose primary sign]
    K --> L[Write JSONL event]
    L --> M[Draw overlay]
    M --> N[Write output video]
'''


In [ ]:
# Mermaid source: Bước 7 - Review output ngay trong notebook
diagram_review_output = r'''flowchart LR
    A[Output files] --> B[Preview frame selection]
    B --> C[Inline HTML gallery]
    C --> D[Embedded video playback]
'''


In [ ]:
# Mermaid source: Bước 8 - Đánh giá model, ODD proxy và confidence

diagram_eval_metrics = r'''flowchart TD
    A[events.jsonl]
    --> B[Frame and feature KPI]
    A --> C[Confidence distribution]
    A --> D[Size bins and track latency]
    A --> E[Threshold sweep]
    F[Optional GT annotations] --> G[Precision Recall IoU50]
    B --> H[ODD proxy summary]
    C --> H
    D --> H
    E --> H
    G --> H
'''


## Bước 1 - Setup runtime và dependency

Mục tiêu của nhóm cell này là làm cho notebook **tự khởi động được trên Colab** mà không giả định môi trường đã cài sẵn package.



Ý nghĩa kỹ thuật:
- giảm lỗi do thiếu package khi mở notebook ở runtime mới;
- làm rõ notebook đang chạy trên Colab hay local Jupyter;
- giúp người review xác định nhanh môi trường trước khi đọc kết quả phía sau.


In [ ]:
# Render diagram: Bước 1 - Setup runtime và dependency
display_mermaid(
    diagram_setup_runtime,
    caption='Luồng setup runtime: phát hiện môi trường, kiểm tra dependency, cài thiếu và in runtime summary.',
)


In [ ]:
# Setup runtime cho Colab hoặc local Jupyter
from __future__ import annotations

import importlib.util
import os
import platform
import subprocess
import sys

# Cờ môi trường: dùng để rẽ nhánh hành vi upload hoặc mount file sau này.
IN_COLAB = 'google.colab' in sys.modules

REQUIRED_PACKAGES = {
    'ultralytics': 'ultralytics>=8.3.0',
    'cv2': 'opencv-python-headless>=4.8.0',
    'huggingface_hub': 'huggingface_hub>=0.23.0',
    'psutil': 'psutil>=5.9.0',
    'requests': 'requests>=2.31.0',
    'gdown': 'gdown>=5.2.0',
}


def ensure_packages(required: dict[str, str]) -> None:
    # Notebook tự kiểm tra dependency để tránh fail ngay ở cell đầu tiên.
    missing = []
    for module_name, pip_name in required.items():
        if importlib.util.find_spec(module_name) is None:
            missing.append(pip_name)
    if missing:
        cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + missing
        print('Installing missing packages:', ', '.join(missing))
        subprocess.check_call(cmd)
    else:
        print('All required packages already available.')


ensure_packages(REQUIRED_PACKAGES)

print('Python      :', sys.version.split()[0])
print('Platform    :', platform.platform())
print('In Colab    :', IN_COLAB)


In [ ]:
# Kiểm tra tài nguyên và chọn profile demo theo RAM/GPU
from pathlib import Path
import shutil

import psutil
import subprocess

# Đo tổng RAM khả dụng để chọn profile phù hợp thay vì chạy một cấu hình cố định.
TOTAL_RAM_GB = psutil.virtual_memory().total / (1024 ** 3)

# Không phải mọi Colab runtime đều có GPU hoặc có sẵn binary nvidia-smi.
# Vì vậy cần fallback sạch về CPU-only thay vì raise FileNotFoundError.
gpu_probe_cmd = shutil.which('nvidia-smi')
if gpu_probe_cmd:
    try:
        gpu_probe = subprocess.run(
            [gpu_probe_cmd, '--query-gpu=name', '--format=csv,noheader'],
            capture_output=True,
            text=True,
            check=False,
        )
        GPU_AVAILABLE = gpu_probe.returncode == 0 and bool(gpu_probe.stdout.strip())
        GPU_NAME = gpu_probe.stdout.strip().splitlines()[0] if GPU_AVAILABLE else 'CPU only'
    except Exception:
        GPU_AVAILABLE = False
        GPU_NAME = 'CPU only'
else:
    GPU_AVAILABLE = False
    GPU_NAME = 'CPU only'

# Các profile này là cách notebook degrade có kiểm soát khi tài nguyên hạn chế.
PROFILES = {
    'high': {
        'imgsz': 640,
        'max_width': 1600,
        'skip': 0,
        'max_frames': None,
        'conf': 0.15,
    },
    'medium': {
        'imgsz': 640,
        'max_width': 1280,
        'skip': 0,
        'max_frames': None,
        'conf': 0.15,
    },
    'low': {
        'imgsz': 512,
        'max_width': 960,
        'skip': 1,
        'max_frames': None,
        'conf': 0.15,
    },
}

# Mặc định notebook sẽ chạy hết video.
# Nếu cần debug nhanh, có thể đặt FRAME_LIMIT_OVERRIDE thành số frame cụ thể.
RUN_FULL_VIDEO = True
FRAME_LIMIT_OVERRIDE = None

if TOTAL_RAM_GB >= 20:
    PROFILE_NAME = 'high'
elif TOTAL_RAM_GB >= 10:
    PROFILE_NAME = 'medium'
else:
    PROFILE_NAME = 'low'

PROFILE = PROFILES[PROFILE_NAME].copy()

if FRAME_LIMIT_OVERRIDE is not None:
    PROFILE['max_frames'] = int(FRAME_LIMIT_OVERRIDE)
elif RUN_FULL_VIDEO:
    PROFILE['max_frames'] = None

print(f'Total RAM   : {TOTAL_RAM_GB:.2f} GB')
print('GPU         :', GPU_NAME)
print('Profile     :', PROFILE_NAME)
print('Parameters  :', PROFILE)
print('Run full video :', RUN_FULL_VIDEO)
print('Frame limit    :', PROFILE['max_frames'])

if TOTAL_RAM_GB < 4:
    print('WARNING: RAM dưới 4 GB. Notebook vẫn có thể chạy hết video, nhưng thời gian xử lý sẽ tăng đáng kể.')


## Bước 2 - Chọn profile theo RAM và nguồn video đầu vào

Nhóm cell này quyết định notebook sẽ chạy ở cấu hình nào và lấy video từ đâu.



Điểm quan trọng:
- `profile` là cơ chế `degraded operation` ở mức notebook: máy yếu thì giảm `max_width`, có thể tăng `skip`, nhưng mặc định notebook vẫn chạy hết video;
- input video được ưu tiên theo thứ tự: repo local, file có sẵn trên Colab, upload thủ công, video demo từ Google Drive, rồi mới đến smoke-test clip;
- video dài hơn làm tổng thời gian chạy tăng gần tuyến tính theo số frame vì detector, preprocess, tracking, overlay và video writer đều lặp lại trên từng frame;
- pipeline hiện không chủ động tích lũy toàn bộ lịch sử video trong RAM: `events.jsonl` được stream ra file theo từng frame, còn danh sách `tracks` chỉ giữ các sign còn sống và sẽ bị prune khi `REJECTED` hoặc `EXPIRED`.

Ghi chú: chỉ nên đặt `FRAME_LIMIT_OVERRIDE` khi cần debug nhanh. Với kịch bản TSR automotive, hành vi mặc định đúng là chạy toàn bộ clip để quan sát đủ state flow của feature.


In [ ]:
# Render diagram: Bước 2 - Chọn profile theo RAM và nguồn video đầu vào
display_mermaid(
    diagram_profile_input,
    caption='Luồng chọn profile theo RAM và resolve input video trước khi chạy pipeline.',
)


In [ ]:
# Chọn input video và vị trí output
import json
import urllib.request
from pathlib import Path

# Điền đường dẫn nếu muốn ép notebook dùng đúng một video cụ thể.
FORCE_VIDEO_PATH = ''
ALLOW_COLAB_UPLOAD = False
DRIVE_VIDEO_URL = 'https://drive.google.com/file/d/1jvMYLCpHR8tJc-bLMggJFDf55-uaAcx_/view?usp=drive_link'
PREFER_DRIVE_DEMO_IF_NO_LOCAL_VIDEO = True
FALLBACK_SMOKE_TEST_URL = 'https://download.samplelib.com/mp4/sample-5s.mp4'

ROOT_CANDIDATES = [
    Path.cwd(),
    Path('/content/adas-tsr'),
    Path('/content/drive/MyDrive/adas-tsr'),
]


def find_repo_root() -> Path | None:
    # Tìm repo root để notebook dùng được cả trong repo local lẫn Colab mount Drive.
    for cand in ROOT_CANDIDATES:
        current = cand.resolve()
        while True:
            if (current / 'code').exists() and (current / 'research').exists():
                return current
            if current == current.parent:
                break
            current = current.parent
    return None


REPO_ROOT = find_repo_root()
if REPO_ROOT is not None:
    print('Repo root detected:', REPO_ROOT)
else:
    print('Repo root not detected. Notebook will run in standalone mode.')

OUTPUT_DIR = (REPO_ROOT / 'research' / 'assets' / 'videos' / 'generated' / 'colab_demo') if REPO_ROOT else Path('/content/tsr_colab_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def maybe_upload_video() -> Path | None:
    if not IN_COLAB or not ALLOW_COLAB_UPLOAD:
        return None
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        return None
    first_name = next(iter(uploaded.keys()))
    return Path(first_name).resolve()


def maybe_download_drive_demo(output_dir: Path) -> Path | None:
    if not PREFER_DRIVE_DEMO_IF_NO_LOCAL_VIDEO:
        return None
    try:
        import gdown
    except Exception as exc:
        print(f'Cannot import gdown for Drive download: {exc}')
        return None
    out_path = output_dir / 'drive_demo_video.mp4'
    if out_path.exists():
        return out_path
    try:
        print('No local input video found. Downloading demo video from Google Drive...')
        gdown.download(url=DRIVE_VIDEO_URL, output=str(out_path), fuzzy=True, quiet=False)
        return out_path if out_path.exists() else None
    except Exception as exc:
        print(f'Google Drive demo download failed: {exc}')
        return None


video_candidates = []
if FORCE_VIDEO_PATH:
    video_candidates.append(Path(FORCE_VIDEO_PATH).expanduser())
if REPO_ROOT:
    video_candidates.extend([
        REPO_ROOT / 'videos' / 'traffic_sign_test.mp4',
        REPO_ROOT / 'videos' / '14212806_3840_2160_60fps.mp4',
        REPO_ROOT / 'videos' / '14212806_demo_output_fixed.mp4',
    ])
video_candidates.extend([
    Path('/content/input.mp4'),
    Path('/content/video.mp4'),
])

INPUT_VIDEO = next((p for p in video_candidates if p.exists()), None)
if INPUT_VIDEO is None:
    INPUT_VIDEO = maybe_upload_video()
if INPUT_VIDEO is None:
    INPUT_VIDEO = maybe_download_drive_demo(OUTPUT_DIR)
if INPUT_VIDEO is None:
    INPUT_VIDEO = OUTPUT_DIR / 'smoke_test.mp4'
    if not INPUT_VIDEO.exists():
        print('No local input video or Drive demo found. Downloading smoke-test clip...')
        urllib.request.urlretrieve(FALLBACK_SMOKE_TEST_URL, INPUT_VIDEO)

print('Input video :', INPUT_VIDEO)
print('Drive demo  :', DRIVE_VIDEO_URL)
print('Output dir  :', OUTPUT_DIR)


## Kiến trúc production-lite được mô phỏng trong notebook



Các block `Lane Association`, `Map Fusion`, `Diagnostics` và `ISA` trong notebook này chỉ được mô phỏng ở mức tối thiểu hoặc dưới dạng stub/policy. Mục tiêu là có được **stateful behavior** đủ rõ để review, không phải thay thế hệ thống trên ECU.


In [ ]:
# Render diagram: Kiến trúc production-lite được mô phỏng trong notebook
display_mermaid(
    diagram_production_lite_arch,
    caption='Kiến trúc production-lite: từ camera frame qua quality gate, ODD gate, detector, tracking, lifecycle và output.',
)


## Bước 3 - Helper layer cho perception và overlay

Đây là lớp hàm tiện ích dùng chung cho toàn notebook. Nó gom các thao tác thường xuyên lặp lại thành từng hàm nhỏ để phần pipeline chính ở dưới dễ đọc hơn.



Cell này chủ yếu giải quyết ba việc:
- chuẩn hóa input/output giữa các bước;
- chuyển class detector thành `sign family` dễ dùng cho policy;
- chuẩn bị logic hiển thị để video output phản ánh đúng `feature state` thay vì chỉ vẽ bbox.


In [ ]:
# Render diagram: Bước 3 - Helper layer cho perception và overlay
display_mermaid(
    diagram_helper_layer,
    caption='Helper layer chuẩn hóa frame, detection, mapping sign family và overlay.',
)


In [ ]:
# Helper functions: detector, quality gate, sign family, overlay
import base64
import json
import re
import time
from collections import Counter
from dataclasses import dataclass
from typing import Any

import cv2
import numpy as np
from huggingface_hub import hf_hub_download
from IPython.display import HTML, Video, display
from ultralytics import YOLO

DEFAULT_HF_REPO = 'star092304/traffic-sign-detection-vietnam-yolo'

SIGN_COLORS = {
    'stop': (0, 0, 255),
    'speed': (0, 0, 255),
    'no_entry': (0, 0, 255),
    'warning': (0, 200, 255),
    'mandatory': (255, 0, 0),
    'info': (0, 180, 0),
    'default': (0, 180, 0),
}

SEVERE_FAMILIES = {'stop', 'no_entry'}
SPEED_FAMILIES = {'speed'}


def preprocess(frame: np.ndarray, max_width: int = 960) -> np.ndarray:
    # Giảm độ phân giải frame quá lớn để giữ runtime và RAM ở mức demo chấp nhận được.
    h, w = frame.shape[:2]
    if w > max_width:
        scale = max_width / float(w)
        frame = cv2.resize(frame, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_LINEAR)
    return frame


def scale_bbox(x1: int, y1: int, x2: int, y2: int, from_shape: tuple[int, int, int], to_shape: tuple[int, int, int]) -> tuple[int, int, int, int]:
    fh, fw = from_shape[:2]
    th, tw = to_shape[:2]
    sx = tw / max(fw, 1)
    sy = th / max(fh, 1)
    return int(x1 * sx), int(y1 * sy), int(x2 * sx), int(y2 * sy)


def classify_family(label: str) -> str:
    # Detector có thể trả nhiều class cụ thể; feature policy chỉ cần vài family chính để quyết định warning level.
    key = label.lower().replace('_', ' ')
    if 'speed' in key or re.search(r'\b(5|10|20|30|40|50|60|70|80|90|100|120)\b', key):
        return 'speed'
    if 'stop' in key:
        return 'stop'
    if 'no entry' in key or 'cấm vào' in key:
        return 'no_entry'
    if any(word in key for word in ['warning', 'danger', 'curve', 'slippery', 'children', 'pedestrian']):
        return 'warning'
    if any(word in key for word in ['turn', 'keep', 'one way', 'mandatory', 'roundabout']):
        return 'mandatory'
    if any(word in key for word in ['parking', 'station', 'information']):
        return 'info'
    return 'default'


def color_for_family(family: str) -> tuple[int, int, int]:
    return SIGN_COLORS.get(family, SIGN_COLORS['default'])


def warning_level_for_family(family: str, confirmed: bool, quality_ok: bool) -> int:
    if not confirmed or not quality_ok:
        return 0
    if family in SEVERE_FAMILIES:
        return 3
    if family in SPEED_FAMILIES:
        return 2
    return 1


def extract_speed_limit(label: str) -> int | None:
    digits = re.findall(r'(?<!\d)(\d{2,3})(?!\d)', label)
    if not digits:
        return None
    value = int(digits[0])
    if 5 <= value <= 130:
        return value
    return None


def iou(box_a: tuple[int, int, int, int], box_b: tuple[int, int, int, int]) -> float:
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - inter + 1e-6
    return inter / union


def assess_frame_quality(frame: np.ndarray) -> dict[str, Any]:
    # Quality gate heuristic: không thay thế production metric, nhưng đủ để mô phỏng lý do suppress/degrade.
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur_var = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    mean_luma = float(gray.mean())
    dark_ratio = float((gray < 35).mean())
    bright_ratio = float((gray > 245).mean())

    reasons = []
    if blur_var < 45:
        reasons.append('LOW_SHARPNESS')
    if mean_luma < 45 or dark_ratio > 0.55:
        reasons.append('TOO_DARK')
    if bright_ratio > 0.18:
        reasons.append('GLARE')

    quality_ok = not reasons
    should_infer = 'GLARE' not in reasons and 'TOO_DARK' not in reasons
    return {
        'blur_var': blur_var,
        'mean_luma': mean_luma,
        'dark_ratio': dark_ratio,
        'bright_ratio': bright_ratio,
        'reasons': reasons,
        'quality_ok': quality_ok,
        'should_infer': should_infer,
    }


def detect_with_yolo(frame: np.ndarray, model: YOLO, conf_thres: float = 0.15, imgsz: int = 640) -> list[dict[str, Any]]:
    results = model(frame, imgsz=imgsz, conf=conf_thres, verbose=False, max_det=20)[0]
    detections: list[dict[str, Any]] = []
    names = model.names
    for box in results.boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        label = names.get(cls_id, str(cls_id))
        family = classify_family(label)
        detections.append({
            'bbox': (x1, y1, x2, y2),
            'label': label,
            'conf': round(conf, 4),
            'family': family,
            'speed_limit': extract_speed_limit(label),
        })
    return detections


def draw_overlay(frame: np.ndarray, tracks: list['SignTrack'], quality: dict[str, Any], feature_state: str, fps_value: float | None = None) -> np.ndarray:
    # Overlay hiển thị theo track/state thay vì chỉ theo bbox detector để người xem thấy được feature behavior.
    annotated = frame.copy()
    for track in tracks:
        if track.state not in {'CANDIDATE', 'CONFIRMED', 'ACTIVE', 'STALE'}:
            continue
        x1, y1, x2, y2 = track.bbox
        color = color_for_family(track.family)
        if track.state == 'STALE':
            color = (128, 128, 255)
        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
        text = f"T{track.track_id} {track.label} | {track.state} | L{track.warning_level} | {track.confidence:.2f}"
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
        cv2.rectangle(annotated, (x1, max(0, y1 - th - 6)), (x1 + tw + 4, y1), color, -1)
        cv2.putText(annotated, text, (x1 + 2, max(12, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)

    status_lines = [
        f"Feature: {feature_state}",
        f"QualityOK: {quality['quality_ok']}",
        f"Reasons: {','.join(quality['reasons']) if quality['reasons'] else 'NONE'}",
        f"BlurVar: {quality['blur_var']:.1f} | Luma: {quality['mean_luma']:.1f}",
    ]
    if fps_value is not None:
        status_lines.append(f"Infer FPS: {fps_value:.1f}")

    y = 24
    for line in status_lines:
        cv2.putText(annotated, line, (10, y), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2)
        y += 22

    return annotated


## Bước 4 - Tracking, temporal confirmation và state machine

Đây là phần quan trọng nhất của notebook nếu nhìn theo góc độ `feature logic`.



Cell này biến detector frame-level thành sign có lịch sử theo thời gian.

Các nguyên tắc được mô phỏng ở đây:
- không publish sign mới chỉ vì một frame đơn lẻ;
- có `CANDIDATE`, `ACTIVE`, `STALE`, `EXPIRED` để tránh nhấp nháy HMI;
- `warning level` phụ thuộc vào `sign family`, xác nhận theo thời gian, và chất lượng hiện tại.


In [ ]:
# Render diagram: Bước 4 - Tracking, temporal confirmation và state machine
display_mermaid(
    diagram_tracking_state,
    caption='Luồng feature logic: associate detection, update hit/miss, đổi state và chọn primary sign.',
)


In [ ]:
# Tracking, temporal confirmation, lifecycle và HMI policy
@dataclass
class SignTrack:
    # Đây là object state tối thiểu mà feature cần giữ cho mỗi sign còn sống.
    track_id: int
    label: str
    family: str
    bbox: tuple[int, int, int, int]
    confidence: float
    hit_count: int = 1
    miss_count: int = 0
    state: str = 'CANDIDATE'
    speed_limit: int | None = None
    warning_level: int = 0
    source: str = 'camera'


# Các ngưỡng bên dưới có thể chỉnh để minh họa trade-off giữa false advisory và missed sign.
TRACK_CONFIG = {
    'assoc_iou': 0.20,
    'min_confirm_hits': 3,
    'max_candidate_misses': 2,
    'max_stale_misses': 4,
    'expire_misses': 6,
    'map_speed_limit_kph': None,
    'ego_speed_kph': 50,
    'odd_speed_range_kph': (0, 90),
}


def odd_gate(quality: dict[str, Any], ego_speed_kph: float, speed_range: tuple[int, int]) -> tuple[bool, list[str]]:
    # ODD gate ở mức demo: chỉ kiểm tra speed range và một vài quality trigger mạnh.
    reasons = []
    if not (speed_range[0] <= ego_speed_kph <= speed_range[1]):
        reasons.append('ODD_SPEED_OUT')
    for reason in quality['reasons']:
        if reason in {'GLARE', 'TOO_DARK'}:
            reasons.append('ODD_QUALITY_OUT')
            break
    return len(reasons) == 0, reasons


def feature_state_from_quality(odd_ok: bool, quality: dict[str, Any]) -> str:
    if not odd_ok:
        return 'UNAVAILABLE'
    if quality['quality_ok']:
        return 'AVAILABLE'
    return 'DEGRADED'


def apply_map_fusion(track: SignTrack, map_speed_limit_kph: int | None) -> tuple[str, int | None]:
    if map_speed_limit_kph is None or track.speed_limit is None:
        return 'camera_only', track.speed_limit
    if track.speed_limit == map_speed_limit_kph:
        return 'agreed', track.speed_limit
    if track.confidence >= 0.75 and track.hit_count >= TRACK_CONFIG['min_confirm_hits']:
        return 'camera_override', track.speed_limit
    return 'map_override', map_speed_limit_kph


def update_tracks(existing_tracks: list[SignTrack], detections: list[dict[str, Any]], odd_ok: bool, quality: dict[str, Any], frame_idx: int) -> list[SignTrack]:
    # Hàm này là lõi state machine: associate detection, cập nhật hit/miss, rồi chuyển state.
    unmatched_det_idx = set(range(len(detections)))

    for track in existing_tracks:
        best_idx = None
        best_iou = 0.0
        for det_idx in list(unmatched_det_idx):
            det = detections[det_idx]
            if det['family'] != track.family:
                continue
            score = iou(track.bbox, det['bbox'])
            if score > TRACK_CONFIG['assoc_iou'] and score > best_iou:
                best_idx = det_idx
                best_iou = score

        # Nếu tìm được detection tương ứng, track được làm mới và tăng evidence theo thời gian.
        if best_idx is not None:
            det = detections[best_idx]
            track.bbox = det['bbox']
            track.label = det['label']
            track.confidence = max(track.confidence * 0.7 + det['conf'] * 0.3, det['conf'])
            track.speed_limit = det['speed_limit']
            track.hit_count += 1
            track.miss_count = 0
            unmatched_det_idx.remove(best_idx)
            if track.hit_count >= TRACK_CONFIG['min_confirm_hits']:
                track.state = 'ACTIVE' if odd_ok else 'CONFIRMED'
            else:
                track.state = 'CANDIDATE'
        else:
            # Không match được detection mới: track không mất ngay mà đi qua pha STALE rồi mới EXPIRED.
            track.miss_count += 1
            if track.state == 'CANDIDATE' and track.miss_count > TRACK_CONFIG['max_candidate_misses']:
                track.state = 'REJECTED'
            elif track.state in {'CONFIRMED', 'ACTIVE', 'STALE'} and track.miss_count <= TRACK_CONFIG['max_stale_misses']:
                track.state = 'STALE'
            elif track.miss_count > TRACK_CONFIG['expire_misses']:
                track.state = 'EXPIRED'

        if track.state in {'ACTIVE', 'CONFIRMED', 'STALE'}:
            track.warning_level = warning_level_for_family(track.family, track.hit_count >= TRACK_CONFIG['min_confirm_hits'], odd_ok and quality['quality_ok'])
            track.source, fused_speed = apply_map_fusion(track, TRACK_CONFIG['map_speed_limit_kph'])
            if fused_speed is not None:
                track.speed_limit = fused_speed
        else:
            track.warning_level = 0


    next_track_id = 1 + max((track.track_id for track in existing_tracks), default=0)
    for det_idx in sorted(unmatched_det_idx):
        det = detections[det_idx]
        new_track = SignTrack(
            track_id=next_track_id,
            label=det['label'],
            family=det['family'],
            bbox=det['bbox'],
            confidence=det['conf'],
            speed_limit=det['speed_limit'],
        )
        new_track.warning_level = warning_level_for_family(new_track.family, False, False)
        existing_tracks.append(new_track)
        next_track_id += 1

    # Chỉ giữ các track còn sống; track đã reject/expire bị loại khỏi RAM ngay ở đây.
    alive_tracks = [track for track in existing_tracks if track.state not in {'REJECTED', 'EXPIRED'}]
    return alive_tracks


def choose_primary_track(tracks: list[SignTrack]) -> SignTrack | None:
    candidates = [t for t in tracks if t.state in {'ACTIVE', 'STALE', 'CONFIRMED'}]
    if not candidates:
        return None
    candidates.sort(key=lambda t: (t.warning_level, t.confidence, t.hit_count), reverse=True)
    return candidates[0]


## Bước 5 - Nạp model detector

Cell này cố tình giữ nhỏ và độc lập vì đây là ranh giới giữa `environment` và `perception runtime`.



Mục tiêu là giúp notebook chạy được cả khi mở trong repo local lẫn khi chạy độc lập trên Colab.


In [ ]:
# Render diagram: Bước 5 - Nạp model detector
display_mermaid(
    diagram_load_model,
    caption='Quyết định nạp weight local hay tải từ Hugging Face trước khi khởi tạo YOLO.',
)


In [ ]:
# Load model: ưu tiên file local trong repo, nếu không có thì tải từ Hugging Face
# Ưu tiên weight local để notebook tái sử dụng đúng artifact của repo.
if REPO_ROOT and (REPO_ROOT / 'models' / 'best.pt').exists():
    WEIGHTS_PATH = REPO_ROOT / 'models' / 'best.pt'
else:
    WEIGHTS_PATH = Path(hf_hub_download(repo_id=DEFAULT_HF_REPO, filename='best.pt'))

print('Weights path:', WEIGHTS_PATH)
model = YOLO(str(WEIGHTS_PATH))
print('Class count :', len(model.names))
print('Sample names:', list(model.names.items())[:10])


## Bước 6 - Chạy production-lite pipeline end-to-end

Đây là cell chính của notebook. Toàn bộ logic runtime được ghép lại ở đây.



Cell này cũng là nơi notebook tạo các artifact quan trọng nhất:
- `tsr_colab_production_lite_demo.mp4`;
- `events.jsonl`;
- preview frames để xem nhanh ca tiêu biểu.


In [ ]:
# Render diagram: Bước 6 - Chạy production-lite pipeline end-to-end
display_mermaid(
    diagram_end_to_end,
    caption='Pipeline end-to-end: quality gate, detector, tracking, JSONL log và output video.',
)


In [ ]:
# Chạy production-lite pipeline trên toàn bộ video
# Mở video đầu vào và chuẩn bị output. Đây là điểm bắt đầu của runtime end-to-end.
cap = cv2.VideoCapture(str(INPUT_VIDEO))
if not cap.isOpened():
    raise RuntimeError(f'Cannot open video: {INPUT_VIDEO}')

fps_in = cap.get(cv2.CAP_PROP_FPS)
if not fps_in or fps_in < 1 or fps_in > 240:
    fps_in = 30.0
fps_in = float(fps_in)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames_in_video = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)

output_video = OUTPUT_DIR / 'tsr_colab_production_lite_demo.mp4'
events_jsonl = OUTPUT_DIR / 'events.jsonl'
preview_dir = OUTPUT_DIR / 'preview_frames'
preview_dir.mkdir(parents=True, exist_ok=True)

writer = cv2.VideoWriter(
    str(output_video),
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps_in,
    (width, height),
)
if not writer.isOpened():
    raise RuntimeError('VideoWriter could not be opened.')

tracks: list[SignTrack] = []
frame_idx = 0
fps_smooth = 0.0
published_counter = Counter()
state_counter = Counter()
quality_reason_counter = Counter()
family_counter = Counter()
preview_paths: list[Path] = []
# Không giữ toàn bộ events trong RAM vì video dài chỉ cần stream ra JSONL là đủ.
frame_limit = PROFILE['max_frames'] if PROFILE['max_frames'] is not None else float('inf')
print('Total frames in source video:', total_frames_in_video)
print('Frame limit for this run   :', 'FULL VIDEO' if frame_limit == float('inf') else int(frame_limit))

# Vòng lặp chính: mỗi frame sẽ đi qua quality gate, detector, tracker và output policy.
with open(events_jsonl, 'w', encoding='utf-8') as f_jsonl:
    while frame_idx < frame_limit:
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx += 1
        # Preprocess trước để giảm cost inference trên frame quá lớn.
        proc_frame = preprocess(frame, max_width=PROFILE['max_width'])
        quality = assess_frame_quality(proc_frame)
        odd_ok, odd_reasons = odd_gate(quality, TRACK_CONFIG['ego_speed_kph'], TRACK_CONFIG['odd_speed_range_kph'])
        feature_state = feature_state_from_quality(odd_ok, quality)

        run_inference = PROFILE['skip'] <= 0 or (frame_idx % (PROFILE['skip'] + 1) == 1)
        detections: list[dict[str, Any]] = []
        # Chỉ infer khi profile cho phép và quality chưa rơi vào case nên suppress mạnh.
        if run_inference and quality['should_infer']:
            t0 = time.time()
            proc_dets = detect_with_yolo(proc_frame, model, conf_thres=PROFILE['conf'], imgsz=PROFILE['imgsz'])
            infer_dt = max(time.time() - t0, 1e-6)
            inst_fps = 1.0 / infer_dt
            fps_smooth = 0.85 * fps_smooth + 0.15 * inst_fps if fps_smooth > 0 else inst_fps
            for det in proc_dets:
                x1, y1, x2, y2 = det['bbox']
                det['bbox'] = scale_bbox(x1, y1, x2, y2, proc_frame.shape, frame.shape)
                detections.append(det)
        else:
            infer_dt = None

        # Sau detector, feature logic mới quyết định sign nào còn sống và sign nào đáng publish.
        tracks = update_tracks(tracks, detections, odd_ok, quality, frame_idx)
        primary_track = choose_primary_track(tracks)

        for reason in quality['reasons']:
            quality_reason_counter[reason] += 1
        for track in tracks:
            family_counter[track.family] += 1
            state_counter[track.state] += 1
        if primary_track is not None:
            published_counter[f'L{primary_track.warning_level}:{primary_track.family}'] += 1

        # JSONL event là artifact quan trọng nhất cho replay và giải thích decision của feature.
        event = {
            'frame_idx': frame_idx,
            'feature_state': feature_state,
            'odd_ok': odd_ok,
            'odd_reasons': odd_reasons,
            'quality': quality,
            'detections': [
                {
                    'label': det['label'],
                    'family': det['family'],
                    'conf': det['conf'],
                    'bbox': list(map(int, det['bbox'])),
                    'speed_limit': det['speed_limit'],
                }
                for det in detections
            ],
            'tracks': [
                {
                    'track_id': track.track_id,
                    'label': track.label,
                    'family': track.family,
                    'state': track.state,
                    'warning_level': track.warning_level,
                    'conf': round(float(track.confidence), 4),
                    'hit_count': track.hit_count,
                    'miss_count': track.miss_count,
                    'speed_limit': track.speed_limit,
                    'source': track.source,
                }
                for track in tracks
            ],
            'primary_track_id': primary_track.track_id if primary_track else None,
            'primary_warning_level': primary_track.warning_level if primary_track else 0,
            'primary_label': primary_track.label if primary_track else None,
            'runtime': {
                'fps_smooth': round(float(fps_smooth), 3),
                'run_inference': run_inference,
            },
        }
        f_jsonl.write(json.dumps(event, ensure_ascii=False) + '\n')

        # Overlay cuối cùng phản ánh state machine, không chỉ phản ánh detector output của một frame đơn lẻ.
        annotated = draw_overlay(frame, tracks, quality, feature_state, fps_value=fps_smooth if fps_smooth > 0 else None)
        writer.write(annotated)

        if primary_track is not None and len(preview_paths) < 4 and primary_track.warning_level >= 1:
            preview_path = preview_dir / f'frame_{frame_idx:05d}_track_{primary_track.track_id}.jpg'
            cv2.imwrite(str(preview_path), annotated)
            preview_paths.append(preview_path)

cap.release()
writer.release()

summary = {
    'frames_processed': frame_idx,
    'output_video': str(output_video),
    'events_jsonl': str(events_jsonl),
    'profile': PROFILE,
    'published_counter': dict(published_counter),
    'state_counter': dict(state_counter),
    'quality_reason_counter': dict(quality_reason_counter),
    'family_counter': dict(family_counter),
}

print(json.dumps(summary, ensure_ascii=False, indent=2))


## Bước 7 - Review output ngay trong notebook

Sau khi pipeline chạy xong, notebook hiển thị preview frame và video để người dùng không cần rời notebook mới xem được kết quả.



Mục tiêu là rút ngắn vòng lặp review: chạy xong, nhìn thấy kết quả ngay, rồi mới quyết định có cần điều chỉnh profile, threshold hoặc input video hay không.


In [ ]:
# Render diagram: Bước 7 - Review output ngay trong notebook
display_mermaid(
    diagram_review_output,
    caption='Luồng review: chọn preview frame, dựng gallery và phát video output trong notebook.',
)


In [ ]:
# Tóm tắt kết quả, preview frame và video output
print('Output video :', output_video)
print('Events JSONL :', events_jsonl)
print('Preview count:', len(preview_paths))

# Preview frame giúp review nhanh mà không cần phát hết video output.
if preview_paths:
    def make_img_tag(path: Path, width: int = 320) -> str:
        raw = path.read_bytes()
        b64 = base64.b64encode(raw).decode('ascii')
        return f'<div style="margin:6px"><img width="{width}" src="data:image/jpeg;base64,{b64}"><br><small>{path.name}</small></div>'

    html = '<div style="display:flex;flex-wrap:wrap">' + ''.join(make_img_tag(p) for p in preview_paths) + '</div>'
    display(HTML(html))
else:
    print('No preview frames captured. This can happen when the input video has no detectable traffic signs.')

# Nếu notebook hỗ trợ embed video thì phát trực tiếp; nếu không thì in ra đường dẫn file.
try:
    display(Video(str(output_video), embed=True, width=960))
except Exception as exc:
    print('Video preview fallback:', exc)
    print('Open the file manually from:', output_video)


## Bước 8 - Đánh giá model, ODD proxy và confidence

Nhóm cell này trả lời ba câu hỏi thực dụng sau khi pipeline chạy xong:
- feature đã hoạt động trong phần nào của `ODD` quan sát được trên clip;
- confidence score của detector phân bố ra sao theo `family` và theo kích thước biển;
- nếu có ground truth, precision/recall ở `IoU=0.5` đang ở mức nào.

Lưu ý quan trọng:
- không có ground truth thì notebook **không** thể kết luận precision/recall hay mAP thật;
- nhưng vẫn có thể tính được các `proxy metric` rất hữu ích cho demo và debug: `AVAILABLE/DEGRADED/UNAVAILABLE ratio`, quality reason ratio, confidence distribution, size-bin coverage, threshold sweep và confirmation latency.


In [ ]:
# Render diagram: Bước 8 - Đánh giá model, ODD proxy và confidence

display_mermaid(
    diagram_eval_metrics,
    caption='Luồng đánh giá: đọc events.jsonl để suy ra KPI frame/feature, confidence, size coverage và optional precision-recall nếu có ground truth.',
)


In [ ]:
# Đánh giá model: ODD proxy, confidence score, size coverage và optional GT
from collections import Counter, defaultdict
from html import escape
from pathlib import Path
import json

GT_JSON_PATH = None  # Ví dụ: OUTPUT_DIR / 'gt_annotations.json'
GT_MATCH_MODE = 'family'  # 'family' hoặc 'label'
GT_IOU_THRESHOLD = 0.5
CONF_THRESH_SWEEP = sorted({round(float(PROFILE['conf']), 2), 0.25, 0.35, 0.50, 0.70})


def load_events_jsonl(path: Path) -> list[dict[str, Any]]:
    events = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                events.append(json.loads(line))
    return events


def to_pct(numerator: int | float, denominator: int | float) -> float:
    return 100.0 * float(numerator) / float(denominator) if denominator else 0.0


def percentile(values: list[float], q: float) -> float | None:
    if not values:
        return None
    return float(np.percentile(np.asarray(values, dtype=float), q))


def summarize_range(values: list[float]) -> str:
    if not values:
        return '-'
    p10 = percentile(values, 10)
    p50 = percentile(values, 50)
    p90 = percentile(values, 90)
    return f'p10={p10:.1f}, p50={p50:.1f}, p90={p90:.1f}'


def bbox_size_metrics(bbox: list[int] | tuple[int, int, int, int]) -> tuple[int, int, int, int]:
    x1, y1, x2, y2 = map(int, bbox)
    w = max(0, x2 - x1)
    h = max(0, y2 - y1)
    area = w * h
    short_side = min(w, h)
    return w, h, area, short_side


def size_bin_for_bbox(bbox: list[int] | tuple[int, int, int, int]) -> str:
    _, _, _, short_side = bbox_size_metrics(bbox)
    if short_side < 16:
        return '0-15 px'
    if short_side < 32:
        return '16-31 px'
    if short_side < 64:
        return '32-63 px'
    return '64+ px'


def html_table(rows: list[dict[str, Any]], columns: list[str]) -> str:
    if not rows:
        return '<p><i>Không có dữ liệu.</i></p>'
    head = ''.join(f'<th style="text-align:left;padding:6px 8px;border-bottom:1px solid #ccc">{escape(col)}</th>' for col in columns)
    body_rows = []
    for row in rows:
        tds = []
        for col in columns:
            value = row.get(col, '')
            tds.append(f'<td style="padding:6px 8px;border-bottom:1px solid #eee">{escape(str(value))}</td>')
        body_rows.append('<tr>' + ''.join(tds) + '</tr>')
    return '<table style="border-collapse:collapse;width:100%"><thead><tr>' + head + '</tr></thead><tbody>' + ''.join(body_rows) + '</tbody></table>'


def match_key(obj: dict[str, Any], mode: str) -> str:
    if mode == 'label':
        return str(obj.get('label', ''))
    family = obj.get('family')
    if family:
        return str(family)
    return classify_family(str(obj.get('label', '')))


def iou_from_records(a: dict[str, Any], b: dict[str, Any]) -> float:
    return iou(tuple(map(int, a['bbox'])), tuple(map(int, b['bbox'])))


def load_optional_gt(path_like: str | Path | None) -> dict[int, list[dict[str, Any]]]:
    if not path_like:
        return {}
    path = Path(path_like)
    if not path.exists():
        raise FileNotFoundError(f'GT file not found: {path}')
    payload = json.loads(path.read_text(encoding='utf-8'))
    if isinstance(payload, dict) and 'frames' in payload:
        frames = payload['frames']
    elif isinstance(payload, list):
        frames = payload
    else:
        raise ValueError('GT JSON phải là list hoặc dict có key frames')

    gt_by_frame: dict[int, list[dict[str, Any]]] = defaultdict(list)
    for item in frames:
        frame_idx = int(item['frame_idx'])
        objects = item.get('objects') or item.get('detections') or item.get('gt') or []
        for obj in objects:
            bbox = list(map(int, obj['bbox']))
            label = obj.get('label', obj.get('class', 'unknown'))
            family = obj.get('family') or classify_family(str(label))
            gt_by_frame[frame_idx].append({
                'bbox': bbox,
                'label': label,
                'family': family,
            })
    return gt_by_frame


def evaluate_against_gt(events: list[dict[str, Any]], gt_by_frame: dict[int, list[dict[str, Any]]], conf_threshold: float, match_mode: str, iou_threshold: float) -> dict[str, Any]:
    tp = 0
    fp = 0
    fn = 0
    gt_size_counter = Counter()
    gt_size_hit_counter = Counter()
    family_tp = Counter()
    family_fp = Counter()
    family_fn = Counter()

    event_by_frame = {int(event['frame_idx']): event for event in events}
    frame_ids = sorted(set(event_by_frame) | set(gt_by_frame))

    for frame_idx in frame_ids:
        preds = [
            pred for pred in event_by_frame.get(frame_idx, {}).get('detections', [])
            if float(pred.get('conf', 0.0)) >= conf_threshold
        ]
        gts = list(gt_by_frame.get(frame_idx, []))
        matched_gt = set()

        preds = sorted(preds, key=lambda item: float(item.get('conf', 0.0)), reverse=True)
        for gt in gts:
            gt_size_counter[size_bin_for_bbox(gt['bbox'])] += 1

        for pred in preds:
            best_idx = None
            best_iou = 0.0
            for gt_idx, gt in enumerate(gts):
                if gt_idx in matched_gt:
                    continue
                if match_key(pred, match_mode) != match_key(gt, match_mode):
                    continue
                score = iou_from_records(pred, gt)
                if score >= iou_threshold and score > best_iou:
                    best_iou = score
                    best_idx = gt_idx
            if best_idx is None:
                fp += 1
                family_fp[match_key(pred, match_mode)] += 1
            else:
                matched_gt.add(best_idx)
                gt = gts[best_idx]
                tp += 1
                family_tp[match_key(gt, match_mode)] += 1
                gt_size_hit_counter[size_bin_for_bbox(gt['bbox'])] += 1

        for gt_idx, gt in enumerate(gts):
            if gt_idx not in matched_gt:
                fn += 1
                family_fn[match_key(gt, match_mode)] += 1

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-9)

    size_rows = []
    for size_bin in ['0-15 px', '16-31 px', '32-63 px', '64+ px']:
        total_gt = gt_size_counter[size_bin]
        hit_gt = gt_size_hit_counter[size_bin]
        size_rows.append({
            'size_bin': size_bin,
            'gt_count': total_gt,
            'matched': hit_gt,
            'recall': f'{to_pct(hit_gt, total_gt):.1f}%' if total_gt else '-',
        })

    family_keys = sorted(set(family_tp) | set(family_fp) | set(family_fn))
    family_rows = []
    for key in family_keys:
        tp_k = family_tp[key]
        fp_k = family_fp[key]
        fn_k = family_fn[key]
        prec_k = tp_k / max(tp_k + fp_k, 1)
        rec_k = tp_k / max(tp_k + fn_k, 1)
        family_rows.append({
            'family_or_label': key,
            'tp': tp_k,
            'fp': fp_k,
            'fn': fn_k,
            'precision': f'{100.0 * prec_k:.1f}%',
            'recall': f'{100.0 * rec_k:.1f}%',
        })

    return {
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'size_rows': size_rows,
        'family_rows': family_rows,
    }


events = load_events_jsonl(events_jsonl)
if not events:
    raise RuntimeError(f'No events found in {events_jsonl}. Run the pipeline cell first.')

frame_count = len(events)
duration_s = frame_count / max(float(fps_in), 1e-6)
feature_state_counter = Counter(event['feature_state'] for event in events)
quality_reason_counter_eval = Counter()
odd_reason_counter_eval = Counter()
detection_records = []
track_state_counter_eval = Counter()
publish_frames = 0
quality_ok_frames = 0
should_infer_frames = 0
odd_ok_frames = 0
blur_all = []
luma_all = []
blur_publish = []
luma_publish = []
track_first_seen = {}
track_first_active = {}
track_best_conf = {}

for event in events:
    quality = event['quality']
    blur_all.append(float(quality['blur_var']))
    luma_all.append(float(quality['mean_luma']))
    if quality.get('quality_ok'):
        quality_ok_frames += 1
    if quality.get('should_infer'):
        should_infer_frames += 1
    if event.get('odd_ok'):
        odd_ok_frames += 1
    for reason in quality.get('reasons', []):
        quality_reason_counter_eval[reason] += 1
    for reason in event.get('odd_reasons', []):
        odd_reason_counter_eval[reason] += 1

    if event.get('primary_track_id') is not None:
        publish_frames += 1
        blur_publish.append(float(quality['blur_var']))
        luma_publish.append(float(quality['mean_luma']))

    for det in event.get('detections', []):
        w, h, area, short_side = bbox_size_metrics(det['bbox'])
        detection_records.append({
            'frame_idx': int(event['frame_idx']),
            'family': det['family'],
            'label': det['label'],
            'conf': float(det['conf']),
            'bbox': det['bbox'],
            'width': w,
            'height': h,
            'area': area,
            'short_side': short_side,
            'size_bin': size_bin_for_bbox(det['bbox']),
        })

    for track in event.get('tracks', []):
        track_state_counter_eval[track['state']] += 1
        tid = int(track['track_id'])
        track_first_seen.setdefault(tid, int(event['frame_idx']))
        track_best_conf[tid] = max(track_best_conf.get(tid, 0.0), float(track.get('conf', 0.0)))
        if track['state'] == 'ACTIVE' and tid not in track_first_active:
            track_first_active[tid] = int(event['frame_idx'])

overview_rows = [
    {'metric': 'Frames processed', 'value': frame_count},
    {'metric': 'Video duration observed', 'value': f'{duration_s:.2f} s'},
    {'metric': 'Frames with detections', 'value': f"{sum(1 for event in events if event.get('detections'))} ({to_pct(sum(1 for event in events if event.get('detections')), frame_count):.1f}%)"},
    {'metric': 'Frames with publish decision', 'value': f'{publish_frames} ({to_pct(publish_frames, frame_count):.1f}%)'},
    {'metric': 'Quality OK frames', 'value': f'{quality_ok_frames} ({to_pct(quality_ok_frames, frame_count):.1f}%)'},
    {'metric': 'ODD OK frames', 'value': f'{odd_ok_frames} ({to_pct(odd_ok_frames, frame_count):.1f}%)'},
    {'metric': 'Frames allowed to infer', 'value': f'{should_infer_frames} ({to_pct(should_infer_frames, frame_count):.1f}%)'},
    {'metric': 'Raw detections', 'value': len(detection_records)},
]

feature_rows = [
    {'feature_state': state, 'frames': count, 'ratio': f'{to_pct(count, frame_count):.1f}%'}
    for state, count in sorted(feature_state_counter.items())
]

quality_rows = [
    {'reason': reason, 'count': count, 'ratio_per_frame': f'{to_pct(count, frame_count):.1f}%'}
    for reason, count in quality_reason_counter_eval.most_common()
] or [{'reason': 'NONE', 'count': 0, 'ratio_per_frame': '0.0%'}]

odd_rows = [
    {'odd_reason': reason, 'count': count, 'ratio_per_frame': f'{to_pct(count, frame_count):.1f}%'}
    for reason, count in odd_reason_counter_eval.most_common()
] or [{'odd_reason': 'NONE', 'count': 0, 'ratio_per_frame': '0.0%'}]

oddenv_rows = [
    {'slice': 'All frames', 'blur_var': summarize_range(blur_all), 'mean_luma': summarize_range(luma_all)},
    {'slice': 'Published frames', 'blur_var': summarize_range(blur_publish), 'mean_luma': summarize_range(luma_publish)},
]

family_counter = Counter(det['family'] for det in detection_records)
family_conf_map = defaultdict(list)
for det in detection_records:
    family_conf_map[det['family']].append(det['conf'])
family_rows = []
for family, count in family_counter.most_common():
    vals = family_conf_map[family]
    family_rows.append({
        'family': family,
        'detections': count,
        'mean_conf': f'{float(np.mean(vals)):.3f}',
        'p50_conf': f'{percentile(vals, 50):.3f}',
        'p90_conf': f'{percentile(vals, 90):.3f}',
    })

size_counter = Counter(det['size_bin'] for det in detection_records)
size_conf_map = defaultdict(list)
for det in detection_records:
    size_conf_map[det['size_bin']].append(det['conf'])
size_rows = []
for size_bin in ['0-15 px', '16-31 px', '32-63 px', '64+ px']:
    vals = size_conf_map.get(size_bin, [])
    size_rows.append({
        'size_bin': size_bin,
        'detections': size_counter.get(size_bin, 0),
        'ratio': f'{to_pct(size_counter.get(size_bin, 0), len(detection_records)):.1f}%' if detection_records else '0.0%',
        'mean_conf': f'{float(np.mean(vals)):.3f}' if vals else '-',
        'p90_conf': f'{percentile(vals, 90):.3f}' if vals else '-',
    })

threshold_rows = []
for threshold in CONF_THRESH_SWEEP:
    kept = [det for det in detection_records if det['conf'] >= threshold]
    frames_kept = len({det['frame_idx'] for det in kept})
    threshold_rows.append({
        'conf_threshold': f'{threshold:.2f}',
        'detections_kept': len(kept),
        'frame_coverage': f'{to_pct(frames_kept, frame_count):.1f}%',
        'mean_conf': f'{float(np.mean([det["conf"] for det in kept])):.3f}' if kept else '-',
    })

latency_frames = [track_first_active[tid] - first_seen for tid, first_seen in track_first_seen.items() if tid in track_first_active]
track_rows = [
    {'metric': 'Unique tracks observed', 'value': len(track_first_seen)},
    {'metric': 'Tracks reaching ACTIVE', 'value': len(track_first_active)},
    {'metric': 'Confirmation latency', 'value': summarize_range([float(v) for v in latency_frames]) if latency_frames else '-'},
    {'metric': 'Best track confidence', 'value': summarize_range(list(track_best_conf.values())) if track_best_conf else '-'},
]
state_rows = [
    {'track_state': state, 'count': count}
    for state, count in track_state_counter_eval.most_common()
]

html_parts = []
html_parts.append('<h3>1. Tổng quan replay và feature state</h3>')
html_parts.append(html_table(overview_rows, ['metric', 'value']))
html_parts.append('<h3>2. ODD proxy và quality gate</h3>')
html_parts.append(html_table(feature_rows, ['feature_state', 'frames', 'ratio']))
html_parts.append(html_table(quality_rows, ['reason', 'count', 'ratio_per_frame']))
html_parts.append(html_table(odd_rows, ['odd_reason', 'count', 'ratio_per_frame']))
html_parts.append(html_table(oddenv_rows, ['slice', 'blur_var', 'mean_luma']))
html_parts.append('<h3>3. Confidence score theo sign family</h3>')
html_parts.append(html_table(family_rows, ['family', 'detections', 'mean_conf', 'p50_conf', 'p90_conf']))
html_parts.append('<h3>4. Kích thước biển quan sát được</h3>')
html_parts.append(html_table(size_rows, ['size_bin', 'detections', 'ratio', 'mean_conf', 'p90_conf']))
html_parts.append('<h3>5. Threshold sweep sau khi đã chạy detector</h3>')
html_parts.append(html_table(threshold_rows, ['conf_threshold', 'detections_kept', 'frame_coverage', 'mean_conf']))
html_parts.append('<h3>6. Track stability và confirmation latency</h3>')
html_parts.append(html_table(track_rows, ['metric', 'value']))
html_parts.append(html_table(state_rows, ['track_state', 'count']))

display(HTML(''.join(html_parts)))

print('Gợi ý đọc nhanh:')
print('- Nếu Published frames chỉ xuất hiện khi blur_var và mean_luma nằm trong một khoảng hẹp, đó là ODD proxy hiện tại của clip này.')
print('- Nếu detections tập trung ở 32-63 px hoặc 64+ px, model đang ít evidence ở biển rất nhỏ trong clip demo này.')
print('- Threshold sweep chỉ cho biết độ nhạy theo confidence trên output hiện có; nó không thay thế precision/recall thật.')

try:
    gt_by_frame = load_optional_gt(GT_JSON_PATH)
except Exception as exc:
    gt_by_frame = None
    print(f'GT load error: {exc}')

if gt_by_frame:
    gt_result = evaluate_against_gt(
        events=events,
        gt_by_frame=gt_by_frame,
        conf_threshold=float(PROFILE['conf']),
        match_mode=GT_MATCH_MODE,
        iou_threshold=float(GT_IOU_THRESHOLD),
    )
    gt_summary_rows = [
        {'metric': 'TP', 'value': gt_result['tp']},
        {'metric': 'FP', 'value': gt_result['fp']},
        {'metric': 'FN', 'value': gt_result['fn']},
        {'metric': 'Precision', 'value': f"{100.0 * gt_result['precision']:.1f}%"},
        {'metric': 'Recall', 'value': f"{100.0 * gt_result['recall']:.1f}%"},
        {'metric': 'F1', 'value': f"{100.0 * gt_result['f1']:.1f}%"},
    ]
    display(HTML('<h3>7. Ground-truth evaluation (optional)</h3>' + html_table(gt_summary_rows, ['metric', 'value'])))
    display(HTML(html_table(gt_result['size_rows'], ['size_bin', 'gt_count', 'matched', 'recall'])))
    if gt_result['family_rows']:
        display(HTML(html_table(gt_result['family_rows'], ['family_or_label', 'tp', 'fp', 'fn', 'precision', 'recall'])))
else:
    print('
Optional GT evaluation chưa chạy. Muốn tính precision/recall thật, hãy tạo một file JSON như ví dụ dưới đây rồi gán vào GT_JSON_PATH:')
    print(json.dumps({
        'frames': [
            {
                'frame_idx': 1,
                'objects': [
                    {'bbox': [100, 120, 168, 190], 'label': 'speed_limit_50', 'family': 'speed'}
                ],
            }
        ]
    }, ensure_ascii=False, indent=2))


## Cách đọc output của notebook

### `feature_state`
- `AVAILABLE`: quality và ODD đang ổn.
- `DEGRADED`: quality chưa đủ tốt để tin mạnh vào sign mới.
- `UNAVAILABLE`: đang ngoài ODD phase-1 hoặc quality quá xấu.

### `warning_level`
- `Level 0`: chưa đủ điều kiện publish.
- `Level 1`: sign thường, hiển thị thông tin cơ bản.
- `Level 2`: `Speed Limit` đã đủ điều kiện xác nhận.
- `Level 3`: sign có mức ưu tiên cao như `Stop` hoặc `No Entry`.

### `events.jsonl`
Mỗi dòng là một frame event để replay:
- quality reasons;
- detections của frame đó;
- các track đang sống;
- track nào được ưu tiên publish;
- runtime metadata.

File này đặc biệt hữu ích khi muốn giải thích vì sao một sign bị suppress hoặc vì sao HMI không đổi ngay lập tức.


## Giới hạn và hướng mở rộng

Notebook này mới chỉ thực hiện được phần `production-lite` có thể mô phỏng trên Colab:
- quality gate heuristic;
- temporal confirmation;
- lifecycle/state flow;
- warning level;
- HMI policy đơn giản;
- JSONL event logging.

Các bước tiếp theo nếu muốn tiến gần production hơn:
1. thêm lane/context model thật;
2. thêm map input và localization confidence thật;
3. thêm benchmark latency/memory theo artifact ONNX/OpenVINO;
4. mở rộng KPI hiện có sang false advisory rate, missed advisory rate và latency p95/p99;
5. thêm test set có gắn scenario tags theo ODD và ground truth chuẩn hóa để chạy precision/recall thật.
